# AI Model Experiment & Evaluation
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membandingkan dua pendekatan AI dalam menyelesaikan task sentiment analysis ulasan pelanggan:
1. Model klasik Machine Learning (Scikit-learn)
2. LLM API (Gemini)

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

> 🔧 **Catatan Penyesuaian Dataset**
>
> Notebook ini menggunakan dataset `customer_reviews_sentiment.csv` dengan kolom `review_text` dan `sentiment` (nilai: `positif`/`negatif`).
>
> Apabila dataset final berbeda dari yang digunakan saat ini, sesuaikan bagian berikut:
> - Nama file pada `pd.read_csv(...)` di Section 2
> - Nama kolom teks dan label pada Section 3 (saat ini: `review_text`, `sentiment`)
> - Label yang diminta pada prompt LLM di Section 5.2 (saat ini: `positif`/`negatif`)
> - Studi Kasus pada Assignment Brief, apabila domain data berbeda dari e-commerce

## 1. Problem Statement

_Tuliskan pemahamanmu terhadap use case di sini (objective, target/label, batasan/asumsi)._

## 2. Import Library & Load Dataset

In [1]:
# 🔧 Sesuaikan nama file jika dataset final berbeda
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

df = pd.read_csv('../data/customer_reviews_sentiment.csv')
print("Ukuran dataset:", df.shape)
df.head()

Ukuran dataset: (200, 4)


,review_id,product_name,review_text,sentiment
0,1,Kemeja Flanel,Pelayanan lambat dan tidak responsif saat diko...,negatif
1,2,Case HP,"Puas banget belanja disini, proses cepat dan b...",positif
2,3,Rice Cooker,"Terima kasih seller, barangnya awet dan sesuai...",positif
3,4,Case HP,Warna produk berbeda jauh dari foto di listing.,negatif
4,5,Headset Bluetooth,"Terima kasih seller, barangnya awet dan sesuai...",positif


In [2]:
# .duplicated() memberi True/False per baris: apakah isinya pernah muncul sebelumnya
# .sum() menjumlahkan True (dihitung 1), jadi hasilnya = jumlah baris duplikat
print("Jumlah review_text yang duplikat:", df['review_text'].duplicated().sum())

# .nunique() = number of unique, hitung berapa banyak nilai yang benar-benar berbeda
print("Jumlah review_text unik:", df['review_text'].nunique())
print("Total baris:", len(df))
print(df['sentiment'].value_counts())

Jumlah review_text yang duplikat: 160
Jumlah review_text unik: 40
Total baris: 200
sentiment
positif    110
negatif     90
Name: count, dtype: int64


## 3. Menyiapkan Train/Test Split

Pastikan test set yang sama digunakan untuk kedua pendekatan agar perbandingan adil.

In [3]:
# 🔧 Sesuaikan nama kolom jika dataset final menggunakan nama kolom berbeda
X = df['review_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Train size:', len(X_train))
print('Test size:', len(X_test))

Train size: 160
Test size: 40


In [4]:
# Cek kebocoran data antara training set dan test set
# set() mengambil nilai unik, & mengambil irisan keduanya
kalimat_bocor = set(X_train) & set(X_test)

print("Jumlah kalimat yang muncul di training DAN test set:", len(kalimat_bocor))
print("Jumlah kalimat unik di test set:", X_test.nunique())

Jumlah kalimat yang muncul di training DAN test set: 25
Jumlah kalimat unik di test set: 26


## 4. Pendekatan 1 — Model Klasik (Scikit-learn)

### 4.1 Preprocessing & Feature Extraction

In [5]:
vectorizer = TfidfVectorizer()
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

In [6]:
# contoh: bagaimana satu ulasan berubah jadi angka TF-IDF
print("Teks asli:")
print(X_train.iloc[0])
print()

# .toarray() mengubah hasil TF-IDF jadi deretan angka biasa
# [0] = baris pertama
angka = X_train_vec[0].toarray()[0]

# tampilkan hanya kata yang nilainya bukan nol
kata = vectorizer.get_feature_names_out()
for i in range(len(angka)):
    if angka[i] > 0:
        print(f"{kata[i]:20s} → {angka[i]:.3f}")

Teks asli:
Pelayanan lambat dan tidak responsif saat dikomplain.

dan                  → 0.242
dikomplain           → 0.481
lambat               → 0.337
pelayanan            → 0.387
responsif            → 0.396
saat                 → 0.481
tidak                → 0.240


### 4.2 Training Model

In [7]:
clf = LogisticRegression()
clf.fit(X_train_vec, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following

### 4.3 Prediksi pada Test Set

In [8]:
y_pred_classic = clf.predict(X_test_vec)

## 5. Pendekatan 2 — LLM API (Gemini)

### 5.1 Setup API

In [9]:
# muat isi file .env ke environment
# API key disimpan di .env, JANGAN hardcode langsung di notebook
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

### 5.2 Merancang Prompt

_prompt : Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata huruf kecil: positif atau negatif.
Jangan beri penjelasan atau tanda baca apa pun, Pendekatan Zero-shot : tugas klasifikasi sederhanya yang sudah ada labelnya, sudah di kuasai model dan lebih hemat token_

In [10]:
def classify_sentiment_llm(review_text, max_percobaan=3):
    prompt = f'''Klasifikasikan sentimen ulasan berikut sebagai "positif" atau "negatif".
Jawab hanya dengan satu kata huruf kecil: positif atau negatif.
Jangan beri penjelasan atau tanda baca apa pun.

Ulasan: "{review_text}"
Sentimen:'''

    # coba beberapa kali kalau server sedang bermasalah
    for percobaan in range(max_percobaan):
        try:
            response = client.models.generate_content(
                model="gemini-3.8-flash",
                contents=prompt
            )
            return response.text.strip().lower()
        except Exception as e:
            # kalau ini percobaan terakhir, menyerah dan teruskan error-nya
            if percobaan == max_percobaan - 1:
                raise
            # kalau belum, tunggu 15 detik lalu coba lagi
            time.sleep(15)




### 5.3 Menjalankan Prediksi pada Test Set

Catatan: lakukan normalisasi terhadap output LLM sebelum dibandingkan dengan label asli (misal: lowercase, strip whitespace).

In [11]:
import time
import json

# ============================================================
# KODE PEMANGGILAN GEMINI API
# Dinonaktifkan setelah dijalankan karena kuota tier gratis
# dibatasi 20 permintaan per hari. Hasilnya sudah disimpan ke
# data/gemini_predictions.json agar notebook dapat dijalankan
# ulang tanpa memanggil API.
# ============================================================

# JUMLAH = 20
# y_pred_llm = []
# pesan_error = ""
#
# for i in range(JUMLAH):
#     teks = X_test.iloc[i]
#     berhasil = False
#
#     for percobaan in range(3):
#         try:
#             hasil = classify_sentiment_llm(teks)
#             y_pred_llm.append(hasil)
#             berhasil = True
#             break
#         except Exception as err:
#             pesan_error = str(err)
#             if "PerDay" in pesan_error:
#                 break
#             if percobaan < 2:
#                 time.sleep(15)
#
#     if not berhasil:
#         y_pred_llm.append("error")
#         print(f"Baris {i} gagal: {pesan_error[:80]}")
#         if "PerDay" in pesan_error:
#             print("Kuota harian habis. Proses dihentikan.")
#             break
#
#     time.sleep(13)
#     print(f"{i+1}/{JUMLAH} — {y_pred_llm[-1]}")

# muat hasil prediksi yang sudah tersimpan
with open('../data/gemini_predictions.json', 'r') as f:
    y_pred_llm = json.load(f)

print("Prediksi Gemini dimuat:", len(y_pred_llm))
print(y_pred_llm)

Prediksi Gemini dimuat: 15
['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'negatif', 'positif', 'positif', 'error']


In [12]:
# ambil hanya prediksi yang berhasil (14 pertama)
y_pred_llm_valid = y_pred_llm[:14]

# ambil potongan test set yang sama persis, posisi 0 sampai 13
y_test_subset = y_test[:14]

print("Jumlah prediksi Gemini:", len(y_pred_llm_valid))
print("Jumlah label pembanding:", len(y_test_subset))
print()

from collections import Counter
print("Sebaran prediksi Gemini:", Counter(y_pred_llm_valid))
print("Sebaran label asli     :", Counter(y_test_subset))

Jumlah prediksi Gemini: 14
Jumlah label pembanding: 14

Sebaran prediksi Gemini: Counter({'positif': 8, 'negatif': 6})
Sebaran label asli     : Counter({'positif': 8, 'negatif': 6})


In [13]:
print(y_pred_llm)

['negatif', 'positif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'positif', 'negatif', 'negatif', 'positif', 'negatif', 'positif', 'positif', 'error']


In [14]:
import json

# simpan hasil prediksi Gemini ke file, agar tidak perlu memanggil API lagi
with open('../data/gemini_predictions.json', 'w') as f:
    json.dump(y_pred_llm, f)

print("Tersimpan:", len(y_pred_llm), "prediksi")

Tersimpan: 15 prediksi


## 6. Evaluasi dan Perbandingan

### 6.1 Evaluasi Model Klasik

In [15]:
print('=== Model Klasik (Scikit-learn) ===')
print('Accuracy:', accuracy_score(y_test, y_pred_classic))
print(classification_report(y_test, y_pred_classic))
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred_classic))

=== Model Klasik (Scikit-learn) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00        18
     positif       1.00      1.00      1.00        22

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

Confusion Matrix:
 [[18  0]
 [ 0 22]]


### Interpretasi Hasil Model Klasik

**Accuracy 1.00** — model menebak benar 40 dari 40 ulasan di test set.

**Precision, recall, dan F1-score 1.00** untuk kedua kelas — tidak ada
kesalahan prediksi sama sekali, baik pada kelas positif maupun negatif.

**Confusion matrix** menunjukkan 18 ulasan negatif dan 22 ulasan positif
seluruhnya tertebak benar, tanpa satu pun kesalahan.

**Namun hasil ini tidak dapat dipercaya.** karena 25 dari 26 kalimat test sudah ada di training set, jadi model mengingat bukan memahami. Sehingga kalau ada ulasan baru di luar training set, akurasinya lebih rendah dari 1.00. artinya angka evaluasi  ini tidak reliable untuk di jadikan patokan / data point yang terverifikasi 

### 6.2 Evaluasi LLM API

In [16]:
# evaluasi Gemini pada 14 ulasan
print("=== Model LLM API (Gemini) ===")
print("Accuracy:", accuracy_score(y_test_subset, y_pred_llm_valid))
print(classification_report(y_test_subset, y_pred_llm_valid))
print("Confusion Matrix:\n", confusion_matrix(y_test_subset, y_pred_llm_valid))

print()
print("=== Model Klasik pada 14 ulasan yang sama ===")
# potong prediksi model klasik di posisi yang sama, supaya perbandingannya adil
y_pred_classic_subset = y_pred_classic[:14]
print("Accuracy:", accuracy_score(y_test_subset, y_pred_classic_subset))
print(classification_report(y_test_subset, y_pred_classic_subset))
print("Confusion Matrix:\n", confusion_matrix(y_test_subset, y_pred_classic_subset))

=== Model LLM API (Gemini) ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         6
     positif       1.00      1.00      1.00         8

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14

Confusion Matrix:
 [[6 0]
 [0 8]]

=== Model Klasik pada 14 ulasan yang sama ===
Accuracy: 1.0
              precision    recall  f1-score   support

     negatif       1.00      1.00      1.00         6
     positif       1.00      1.00      1.00         8

    accuracy                           1.00        14
   macro avg       1.00      1.00      1.00        14
weighted avg       1.00      1.00      1.00        14

Confusion Matrix:
 [[6 0]
 [0 8]]


### Interpretasi Hasil Model LLM API (Gemini)

**Keterbatasan jumlah data uji.** Tier gratis Gemini API dibatasi 20 permintaan
per hari, sehingga seluruh 40 ulasan test set tidak dapat diklasifikasikan dalam
satu hari. Proses dijalankan untuk 20 ulasan pertama dan berhenti pada ulasan
ke-14 karena kuota harian habis. Evaluasi dilakukan pada 14 ulasan tersebut.

**Menjaga perbandingan tetap adil.** Model klasik dievaluasi ulang pada 14 ulasan
yang sama persis (posisi yang sama dari test set), dan hasilnya tetap 1.00.

**Accuracy 1.00** — 14 dari 14 ulasan tertebak benar. Precision, recall, dan
F1-score juga 1.00 untuk kedua kelas. Confusion matrix menunjukkan 6 ulasan
negatif dan 8 ulasan positif seluruhnya tertebak benar.

**Berbeda sebabnya dari model klasik.** Gemini mencapai 1.00 tanpa pernah melihat
data latih sama sekali (zero-shot), sehingga angka ini tidak terpengaruh kebocoran
data. Yang membatasi justru datasetnya: kalimat-kalimatnya terlalu jelas sentimennya
sehingga tidak cukup menantang untuk membedakan kedua pendekatan.

**Trade-off yang teramati.** Gemini membutuhkan waktu jauh lebih lama — sekitar 13
detik per ulasan karena batas 5 permintaan per menit — dibandingkan model klasik
yang memproses seluruh test set dalam hitungan milidetik. Selain itu, pada tier
gratis terdapat batas kuota harian yang menghentikan proses di tengah jalan.

### 6.3 Tabel Perbandingan Ringkasan

_Susun tabel ringkasan (bisa markdown table atau DataFrame) yang membandingkan Accuracy, Precision, Recall, F1-Score kedua pendekatan._

| Metrik | Model Klasik | Gemini API |
|---|---|---|
| Accuracy | 1.00 | 1.00 |
| Precision (negatif) | 1.00 | 1.00 |
| Recall (negatif) | 1.00 | 1.00 |
| F1-score (negatif) | 1.00 | 1.00 |
| Precision (positif) | 1.00 | 1.00 |
| Recall (positif) | 1.00 | 1.00 |
| F1-score (positif) | 1.00 | 1.00 |
| Jumlah data uji | 14 | 14 |
| Waktu proses | < 1 detik | ± 3 menit |
| Perlu training | Ya | Tidak |
| Butuh koneksi internet | Tidak | Ya |
| Biaya per prediksi | Nol setelah training | Per token, terus-menerus |

## 7. Analisis Trade-off dan Limitation

_Tuliskan analisis: pendekatan mana yang lebih unggul dari sisi performa, trade-off effort/kecepatan/biaya, dan keterbatasan masing-masing pendekatan._

## 7. Analisis Trade-off dan Limitation

### Performa

Kedua pendekatan menghasilkan metrik identik (accuracy 1.00), sehingga tidak ada
yang unggul secara performa. Namun penyebabnya berbeda:

- **Model klasik** mencapai 1.00 karena sebagian besar kalimat di test set sudah
  muncul di training set, sehingga model mengingat alih-alih memahami.
- **Gemini** mencapai 1.00 tanpa melihat data latih sama sekali, tetapi dibantu
  oleh dataset yang kalimatnya straightforward dan tidak mengandung ambiguitas.

Dengan kata lain, dataset ini terlalu mudah sehingga metrik evaluasi kehilangan
daya pembeda antara kedua pendekatan.

### Trade-off

**Effort implementasi.** Model klasik memerlukan beberapa tahap: split data,
TF-IDF, dan training. Gemini tidak memerlukan training sama sekali — cukup
merancang prompt — sehingga jauh lebih cepat disiapkan.

**Kecepatan.** Model klasik memproses seluruh 40 ulasan dalam hitungan milidetik.
Gemini membutuhkan sekitar 13 detik per ulasan karena batas 5 permintaan per menit.

**Biaya.** Model klasik hanya memerlukan satu kali training, setelah itu prediksi
tidak berbiaya. Gemini menagih per token pada setiap prediksi, sehingga biayanya
terus bertambah seiring jumlah ulasan. Untuk implementasi nyata diperlukan tier
berbayar, karena tier gratis memiliki batas kuota.

### Limitation Model Klasik

- **Data leakage** — 25 dari 26 kalimat unik di test set sudah ada di training set
- Hanya mengenali kata yang pernah dilihat saat training; buta terhadap kosakata
  baru, bahasa gaul, dan sarkasme
- Perlu dilatih ulang secara berkala ketika bahasa pelanggan berubah

### Limitation Gemini API

- Kuota tier gratis: 5 permintaan per menit dan 20 permintaan per hari
- Bergantung pada layanan pihak ketiga — model `gemini-2.0-flash` dihentikan Google
  di tengah pengerjaan, dan server mengembalikan error 503 berulang kali
- Percobaan yang gagal tetap terhitung

## 8. Rekomendasi Technical Approach

_Tuliskan rekomendasi akhir beserta alasannya, berdasarkan hasil eksperimen di atas._

## 8. Rekomendasi Technical Approach

**Rekomendasi: Model klasik (Scikit-learn), dengan syarat perbaikan dataset.**

**Mengapa metrik tidak dapat dijadikan dasar.** Kedua pendekatan menghasilkan
accuracy 1.00 pada test set yang sama, sehingga metrik evaluasi tidak dapat
memisahkan keduanya. Keputusan harus diambil berdasarkan pertimbangan operasional.

**Dasar pertimbangan:**

1. **Biaya jangka panjang.** Model klasik hanya memerlukan satu kali training,
   setelah itu setiap prediksi tidak berbiaya. Gemini menagih per token pada setiap
   prediksi, sehingga biaya terus bertambah seiring volume ulasan.

2. **Kecepatan.** Untuk fitur yang tampil di halaman produk, model klasik memproses
   seluruh test set dalam milidetik, sementara Gemini membutuhkan sekitar 13 detik
   per ulasan pada tier gratis.

3. **Kemandirian sistem.** Selama pengerjaan, model `gemini-2.0-flash` dihentikan
   oleh Google dan server mengembalikan error 503 berulang kali. Model klasik
   berjalan di infrastruktur sendiri dan tidak bergantung pada keputusan penyedia
   layanan.

**Kelemahan dari pilihan ini.** Model klasik hanya mengenali kata yang pernah
dilihatnya saat training, sehingga lemah menghadapi kosakata baru, bahasa gaul,
dan sarkasme. Pada aspek ini Gemini jelas lebih unggul.

**Syarat yang harus dipenuhi:**

- Dataset diperluas dengan ulasan pelanggan asli yang lebih beragam
- Tidak ada duplikasi kalimat antara training set dan test set
- Model dilatih ulang secara berkala mengikuti perkembangan bahasa pelanggan
- Dilakukan evaluasi ulang pada dataset yang lebih menantang, karena hasil
  eksperimen ini belum mencerminkan performa sebenarnya